# ==============================================================================
# CUADERNO: Caracterización PVT Automatizada para Circuito Self-Bias (Sky130)
# ARCHIVO: self_bias_pvt.ipynb
# DISEÑO OBJETIVO: /home/luigy/eda/designs/OSCAR/xschem/Iref/self_bias_tb.sch
# ==============================================================================
#
# DESCRIPCIÓN GENERAL:
# Este cuaderno realiza un análisis exhaustivo, automatizado y paralelizado de 
# variabilidad por Proceso, Voltaje y Temperatura (PVT) para el circuito de 
# polarización autónoma (Self-Bias / Iref) diseñado en Sky130A (`self_bias_tb.sch`).
#
# OBJETIVOS DE CARACTERIZACIÓN:
# 1. Verificar el arranque correcto del circuito (Startup) sin estados metaestables en 0V.
# 2. Evaluar la estabilidad del voltaje de referencia ($V_R$) y de la corriente de 
#    polarización ($I_{bias}$ de Mp4) frente a variaciones térmicas (-40°C a 125°C).
# 3. Medir la regulación de línea (sensibilidad ante cambios de VDD entre 1.62V y 1.98V).
# 4. Cuantificar el impacto de las esquinas de proceso Sky130 (tt, ss, ff, sf, fs).
# 5. Comprobar que todos los transistores (Mn1, Mn2, Mp3, Mp4, Ms1, Ms2, Ms3) permanezcan 
#    en sus regiones de operación previstas (márgenes de saturación $V_{ds} - V_{dsat} > 0$).
#
# METODOLOGÍA:
# - Ejecución multinúcleo concurrente mediante ThreadPoolExecutor.
# - Aislamiento de carpetas de simulación por hilo para evitar colisiones en archivos .raw.
# - Extracción y consolidación automática de métricas mediante PyLTSpice y Pandas.
# - Generación de reportes numéricos (Excel/CSV) y gráficos de alta resolución (300 DPI).
# ==============================================================================


In [1]:
# ==============================================================================
# ARCHIVO: self_bias_pvt.ipynb - Celda 1: Configuración de Dependencias y Entorno
# DESCRIPCIÓN: 
# Esta celda prepara el entorno de ejecución en Python. Registra las rutas de 
# librerías de usuario (compatibles con Docker y host local), comprueba la 
# disponibilidad de 'PyLTSpice' para la lectura directa de archivos .raw de Ngspice 
# e instala automáticamente la librería en caso de no encontrarse presente.
# ==============================================================================

import site
import sys
import subprocess
import os

# 1. Registrar rutas de paquetes del usuario para entornos contenedores y host
rutas_usuario = [
    '/headless/.local/lib/python3.12/site-packages',
    os.path.expanduser('~/.local/lib/python3.12/site-packages'),
    os.path.expanduser('~/.local/lib/python3.11/site-packages'),
    os.path.expanduser('~/.local/lib/python3.10/site-packages')
]

for ruta in rutas_usuario:
    if os.path.exists(ruta):
        site.addsitedir(ruta)

# 2. Verificar e importar PyLTSpice; si no existe, instalarla en el espacio de usuario
try:
    from PyLTSpice import RawRead
    print("✅ ¡PyLTSpice ya se encuentra instalado y listo para su uso!")
except ImportError:
    print("⚠️ PyLTSpice no fue detectado. Procediendo con la instalación automática...")
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'PyLTSpice'])
        for ruta in rutas_usuario:
            if os.path.exists(ruta):
                site.addsitedir(ruta)
        from PyLTSpice import RawRead
        print("✅ ¡PyLTSpice instalado e importado exitosamente!")
    except Exception as error:
        print(f"❌ Error durante la instalación de PyLTSpice: {error}")
        print("Por favor, instálelo manualmente ejecutando: pip install PyLTSpice")



✅ ¡PyLTSpice ya se encuentra instalado y listo para su uso!


In [2]:
# ==============================================================================
# ARCHIVO: self_bias_pvt.ipynb - Celda 2: Función de Simulación Paralela por Hilo
# DESCRIPCIÓN:
# Define la función 'one_pvt_parallel' encargada de simular una combinación 
# específica de Proceso, Voltaje y Temperatura (PVT). Aísla la ejecución en una 
# subcarpeta individual, modifica dinámicamente los parámetros del netlist SPICE, 
# ejecuta Ngspice en segundo plano por lotes, y extrae los datos escalares y las 
# formas de onda utilizando expresiones regulares y la librería PyLTSpice.
# ==============================================================================

import os
import re
import subprocess
import shutil
import concurrent.futures
import itertools
from PyLTSpice import RawRead
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Diccionarios globales para consolidar los resultados de todas las simulaciones
parametros_self_bias = {}
graficas_self_bias = {}

def one_pvt_parallel(netlist_path, raw_path, temp, vdd, proces):
    """
    Ejecuta una simulación PVT individual para el circuito Self-Bias de manera aislada y segura.
    
    Parámetros:
        netlist_path (str): Ruta base del netlist SPICE generado desde el esquemático.
        raw_path (str): Directorio raíz donde se crearán las carpetas aisladas por hilo.
        temp (float): Temperatura de simulación en grados Celsius (°C).
        vdd (float): Voltaje de alimentación VDD en Voltios (V).
        proces (str): Esquina o 'corner' de modelos Sky130 (tt, ss, ff, sf, fs, etc.).
    """
    # Identificador único y limpio para la combinación PVT (ej: V_1_8_T_27_tt)
    vdd_str = str(vdd).replace('.', '_')
    nombre_llave = f"V_{vdd_str}_T_{temp}_{proces}"

    parametros_self_bias[nombre_llave] = {}
    graficas_self_bias[nombre_llave] = {}

    # 1. CREACIÓN DE LA CARPETA AISLADA PARA ESTE HILO
    # Evita condiciones de carrera entre hilos concurrentes que acceden a archivos con el mismo nombre.
    sim_dir = os.path.join(raw_path, nombre_llave)
    os.makedirs(sim_dir, exist_ok=True)

    directorio_original = os.path.dirname(netlist_path)

    # Lectura del netlist base
    with open(netlist_path, 'r', encoding='utf-8', errors='ignore') as archivo:
        contenido = archivo.read()

    # ====================================================================
    # 2. MODIFICACIÓN DINÁMICA DE PARÁMETROS PVT EN EL NETLIST
    # ====================================================================
    # Sustitución de la esquina de modelos Sky130
    patron_corner = r"(\.lib\s+\S*sky130\.lib\.spice\s+)\w+"
    contenido = re.sub(patron_corner, rf"\g<1>{proces}", contenido)

    # Sustitución de la temperatura de simulación
    patron_temp = r"(\.option\s+temp=)[-\d\.]+"
    contenido = re.sub(patron_temp, rf"\g<1>{temp}", contenido)

    # Sustitución del parámetro de tensión VDD
    patron_vdd = r"(\.param\s+VDD\s*=\s*)[-\d\.]+"
    contenido = re.sub(patron_vdd, rf"\g<1>{vdd}", contenido)

    # ====================================================================
    # 3. REDIRECCIÓN DE ARCHIVOS .RAW A LA CARPETA AISLADA
    # ====================================================================
    # Ruta absoluta para el archivo de punto de operación (OP)
    ruta_op_raw = os.path.join(sim_dir, "self_bias_op.raw")
    contenido = contenido.replace("self_bias_op.raw", ruta_op_raw)

    # Ruta absoluta para el archivo transitorio (TRAN)
    ruta_tran_raw = os.path.join(sim_dir, "self_bias_tran.raw")
    if "self_bias_tran.raw" in contenido:
        contenido = contenido.replace("self_bias_tran.raw", ruta_tran_raw)
    else:
        # Si el netlist original no contiene explícitamente el guardado transitorio,
        # inyectamos el comando 'write' con ruta absoluta justo después del comando 'tran'
        patron_tran = r"(tran\s+[^\n]+)"
        contenido = re.sub(patron_tran, rf"\g<1>\nwrite {ruta_tran_raw} v(vr) v(vdd) v(x1.vgp) v(x1.vdi) v(x1.vdinv)", contenido)

    # Escritura del netlist temporal en el directorio original para respetar los .include relativos
    netlist_temp_path = os.path.join(directorio_original, f"self_bias_temp_{nombre_llave}.spice")
    with open(netlist_temp_path, 'w', encoding='utf-8') as archivo:
        archivo.write(contenido)

    # ====================================================================
    # 4. EJECUCIÓN DE NGSPICE EN MODO POR LOTES (BATCH)
    # ====================================================================
    # Selección robusta de la ruta del ejecutable Ngspice
    ngspice_bin = '/foss/tools/ngspice/bin/ngspice'
    if not os.path.exists(ngspice_bin):
        ngspice_bin = '/foss/tools/bin/ngspice'
        if not os.path.exists(ngspice_bin):
            ngspice_bin = shutil.which('ngspice') or 'ngspice'

    comando = [ngspice_bin, "-b", netlist_temp_path]
    resultado = subprocess.run(comando, capture_output=True, text=True, cwd=directorio_original)

    # Si la simulación falla, limpiar netlist temporal y finalizar hilo
    if resultado.returncode != 0:
        if os.path.exists(netlist_temp_path):
            os.remove(netlist_temp_path)
        print(f"\n⚠️ Falló la simulación para {nombre_llave}: Código {resultado.returncode}")
        return

    # ====================================================================
    # 5. EXTRACCIÓN DE RESULTADOS ESCALARES DESDE LA SALIDA DE CONSOLA
    # ====================================================================
    # Almacenar metadatos PVT de la prueba
    parametros_self_bias[nombre_llave]['temperatura'] = float(temp)
    parametros_self_bias[nombre_llave]['vdd'] = float(vdd)
    parametros_self_bias[nombre_llave]['corner'] = str(proces)

    mi_log = resultado.stdout
    patron_regex = r"([a-zA-Z0-9_]+)\s*=\s*([+-]?\d+\.?\d*[eE]?[+-]?\d*)"
    coincidencias = re.findall(patron_regex, mi_log)

    for nombre, valor in coincidencias:
        parametros_self_bias[nombre_llave][nombre.strip().lower()] = float(valor)

    # Limpiar variable redundante de temperatura nominal si existe
    if "tnom" in parametros_self_bias[nombre_llave]:
        del parametros_self_bias[nombre_llave]["tnom"]

    # ====================================================================
    # 6. EXTRACCIÓN DETALLADA DESDE ARCHIVOS RAW (PyLTSpice)
    # ====================================================================
    # A. Extracción de Punto de Operación (OP)
    if os.path.exists(ruta_op_raw):
        try:
            raw_op = RawRead(ruta_op_raw)
            # Lectura de tensiones de nodos principales
            for nodo in ["v(vr)", "v(vdd)", "v(x1.vgp)", "v(x1.vdi)", "v(x1.vdinv)", "i(v1)"]:
                if nodo in raw_op.get_trace_names():
                    val = float(raw_op.get_trace(nodo).get_wave()[0])
                    nombre_limpio = nodo.replace('v(', '').replace('i(', '').replace(')', '').replace('.', '_')
                    parametros_self_bias[nombre_llave][nombre_limpio] = val

            # Cálculo de la tensión fuente-compuerta del espejo PMOS (Vsgp = VDD - Vgp)
            if 'x1_vgp' in parametros_self_bias[nombre_llave] and 'vdd' in parametros_self_bias[nombre_llave]:
                parametros_self_bias[nombre_llave]['vsgp'] = (
                    parametros_self_bias[nombre_llave]['vdd'] - parametros_self_bias[nombre_llave]['x1_vgp']
                )

            # Extracción de parámetros internos para cada uno de los 7 transistores
            # Mn1, Mn2, Mp3, Mp4 (núcleo self-bias) y Ms1, Ms2, Ms3 (circuito de arranque)
            transistores = [
                ('mn1', 'sky130_fd_pr__nfet_01v8'),
                ('mn2', 'sky130_fd_pr__nfet_01v8'),
                ('mp3', 'sky130_fd_pr__pfet_01v8_lvt'),
                ('mp4', 'sky130_fd_pr__pfet_01v8_lvt'),
                ('ms1', 'sky130_fd_pr__nfet_01v8'),
                ('ms2', 'sky130_fd_pr__pfet_01v8_lvt'),
                ('ms3', 'sky130_fd_pr__nfet_01v8')
            ]

            for inst, modelo in transistores:
                prefijo = f"@m.x1.x{inst}.m{modelo}"
                prefijo_id = f"i(@m.x1.x{inst}.m{modelo}[id])"

                # Extracción de corriente de drenaje
                if prefijo_id in raw_op.get_trace_names():
                    id_val = float(raw_op.get_trace(prefijo_id).get_wave()[0])
                    parametros_self_bias[nombre_llave][f"{inst}_id_ua"] = abs(id_val) * 1e6

                # Extracción de parámetros de pequeña señal y tensiones
                params_mos = ['gm', 'gds', 'vds', 'vdsat', 'vgs', 'vth']
                for p in params_mos:
                    traza_nombre = f"{prefijo}[{p}]"
                    traza_nombre_v = f"v({prefijo}[{p}])"
                    if traza_nombre in raw_op.get_trace_names():
                        val = float(raw_op.get_trace(traza_nombre).get_wave()[0])
                        parametros_self_bias[nombre_llave][f"{inst}_{p}"] = val
                    elif traza_nombre_v in raw_op.get_trace_names():
                        val = float(raw_op.get_trace(traza_nombre_v).get_wave()[0])
                        parametros_self_bias[nombre_llave][f"{inst}_{p}"] = val

                # Cálculo de margen de saturación: Vds_margin = |Vds| - |Vdsat|
                vds_k = f"{inst}_vds"
                vdsat_k = f"{inst}_vdsat"
                if vds_k in parametros_self_bias[nombre_llave] and vdsat_k in parametros_self_bias[nombre_llave]:
                    vds_val = abs(parametros_self_bias[nombre_llave][vds_k])
                    vdsat_val = abs(parametros_self_bias[nombre_llave][vdsat_k])
                    parametros_self_bias[nombre_llave][f"{inst}_vds_margin"] = vds_val - vdsat_val

                # Cálculo de sobretensión: Vov = |Vgs| - |Vth|
                vgs_k = f"{inst}_vgs"
                vth_k = f"{inst}_vth"
                if vgs_k in parametros_self_bias[nombre_llave] and vth_k in parametros_self_bias[nombre_llave]:
                    vgs_val = abs(parametros_self_bias[nombre_llave][vgs_k])
                    vth_val = abs(parametros_self_bias[nombre_llave][vth_k])
                    parametros_self_bias[nombre_llave][f"{inst}_vov"] = vgs_val - vth_val

        except Exception as e_op:
            print(f"\n⚠️ Error al procesar punto de operación OP de {nombre_llave}: {e_op}")

    # B. Extracción de Formas de Onda Transitorias (TRAN)
    if os.path.exists(ruta_tran_raw):
        try:
            raw_tran = RawRead(ruta_tran_raw)
            graficas_self_bias[nombre_llave]["time"] = raw_tran.get_trace("time").get_wave()
            graficas_self_bias[nombre_llave]["vr"] = raw_tran.get_trace("v(vr)").get_wave()
            graficas_self_bias[nombre_llave]["vdd"] = raw_tran.get_trace("v(vdd)").get_wave()

            if "v(x1.vgp)" in raw_tran.get_trace_names():
                graficas_self_bias[nombre_llave]["vgp"] = raw_tran.get_trace("v(x1.vgp)").get_wave()
            if "v(x1.vdinv)" in raw_tran.get_trace_names():
                graficas_self_bias[nombre_llave]["vdinv"] = raw_tran.get_trace("v(x1.vdinv)").get_wave()

        except Exception as e_tran:
            print(f"\n⚠️ Error al leer datos transitorios de {nombre_llave}: {e_tran}")

    # ====================================================================
    # 7. LIMPIEZA DE ARCHIVOS TEMPORALES
    # ====================================================================
    if os.path.exists(netlist_temp_path):
        os.remove(netlist_temp_path)



In [3]:
# ==============================================================================
# ARCHIVO: self_bias_pvt.ipynb - Celda 3: Lanzador Multinúcleo y Barrido PVT
# DESCRIPCIÓN:
# Define el espacio de parámetros PVT a explorar:
# - Temperaturas: Rango industrial completo (-40°C, 27°C, 125°C).
# - Voltajes de alimentación: VDD nominal 1.8V con variación del +/- 10% (1.62V, 1.8V, 1.98V).
# - Esquinas de proceso Sky130: tt, ss, ff, sf, fs (con soporte para corners extendidos).
# Orquesta la ejecución en paralelo mediante ThreadPoolExecutor mostrando el 
# progreso en tiempo real de forma interactiva.
# ==============================================================================

# 1. Definición de Rangos y Esquinas PVT
temperaturas = [-40,125] # [-40, 27, 125]         # Temperaturas en grados Celsius [°C]
voltajes = [1.62,1.98]   #[1.62, 1.80, 1.98]         # Niveles de alimentación VDD [V]

# Esquinas de proceso estándar principales
corners_proceso = corners_criticos = [
    "ss", "ff", "sf", "fs", "ss_hh"
]

# Esquinas de proceso completas (descomentar para análisis exhaustivo de 25 corners):
"""
corners_proceso = [
    "tt", "sf", "ff", "ss", "fs", "ll", "hh", "hl", "lh",
    "sf_ll", "sf_hh", "sf_hl", "sf_lh", "ff_ll", "ff_hh", "ff_hl", "ff_lh",
    "ss_ll", "ss_hh", "ss_hl", "ss_lh", "fs_ll", "fs_hh", "fs_hl", "fs_lh"
]
"""

# 2. Resolución dinámica de rutas (Compatible con Docker y entornos locales)
posibles_netlists = [
    "/headless/.xschem/simulations/self_bias_tb.spice",
    "/foss/designs/OSCAR/xschem/simulation/self_bias_tb.spice",
    os.path.expanduser("~/.xschem/simulations/self_bias_tb.spice"),
    "/home/luigy/eda/designs/OSCAR/xschem/simulation/self_bias_tb.spice",
    os.path.abspath("../xschem/Iref/self_bias_tb.spice")
]

netlist_path = None
for ruta_cand in posibles_netlists:
    if os.path.exists(ruta_cand):
        netlist_path = ruta_cand
        break

if netlist_path is None:
    # Ruta por defecto en entorno Docker
    netlist_path = "/headless/.xschem/simulations/self_bias_tb.spice"
    print(f"ℹ️ Usando ruta predeterminada de netlist: {netlist_path}")
else:
    print(f"✅ Netlist detectado en: {netlist_path}")

# Carpeta base para el almacenamiento aislado de archivos .raw
if os.path.exists("/foss/designs/OSCAR"):
    raw_path = "/foss/designs/OSCAR/xschem/spice_raw_log/raw/"
else:
    raw_path = os.path.expanduser("~/eda/designs/OSCAR/xschem/spice_raw_log/raw/")

os.makedirs(raw_path, exist_ok=True)
print(f"📁 Directorio de resultados raw: {raw_path}")

# 3. Generación del producto cartesiano de combinaciones
todas_las_combinaciones = list(itertools.product(temperaturas, voltajes, corners_proceso))
total_sim = len(todas_las_combinaciones)

print(f"\n🚀 Lanzando {total_sim} simulaciones en PARALELO con ThreadPoolExecutor...")

# 4. Ejecución en paralelo con 5 hilos trabajadores concurrentes
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futuros = [
        executor.submit(one_pvt_parallel, netlist_path, raw_path, temp, vdd, proces)
        for temp, vdd, proces in todas_las_combinaciones
    ]

    completados = 0
    for futuro in concurrent.futures.as_completed(futuros):
        completados += 1
        print(f"Progreso global PVT: {completados}/{total_sim} simulaciones completadas ({(completados/total_sim)*100:.1f}%).", end='\r')

print("\n\n🎉 ¡Simulación masiva paralela finalizada con éxito!")



✅ Netlist detectado en: /headless/.xschem/simulations/self_bias_tb.spice
📁 Directorio de resultados raw: /foss/designs/OSCAR/xschem/spice_raw_log/raw/

🚀 Lanzando 20 simulaciones en PARALELO con ThreadPoolExecutor...
Progreso global PVT: 20/20 simulaciones completadas (100.0%).

🎉 ¡Simulación masiva paralela finalizada con éxito!


In [4]:
# ==============================================================================
# ARCHIVO: self_bias_pvt.ipynb - Celda 4: Reportes Numéricos, Métricas y Exportación
# DESCRIPCIÓN:
# Consolida todos los parámetros en DataFrames de Pandas, calcula estadísticas 
# clave (mínimo, máximo, media, desviación típica) filtrando únicamente columnas 
# numéricas para evitar errores de tipo con cadenas de texto (como 'corner'), 
# calcula figuras de mérito avanzadas para el Self-Bias (Coeficiente Térmico 
# en ppm/°C, Regulación de Línea en mV/V, márgenes de saturación) y exporta 
# los resultados a Excel o CSV.
# ==============================================================================

# 1. Creación del DataFrame consolidado con todos los corners
df_resultados = pd.DataFrame.from_dict(parametros_self_bias, orient='index')

# Ordenar columnas colocando los parámetros de control al inicio
cols_prioritarias = ['temperatura', 'vdd', 'corner', 'vr', 'ibias', 'vr_tran', 'vsgp', 'x1_vgp']
otras_cols = [c for c in df_resultados.columns if c not in cols_prioritarias]
cols_ordenadas = [c for c in cols_prioritarias if c in df_resultados.columns] + otras_cols
df_resultados = df_resultados[cols_ordenadas]

# 2. Resumen estadístico (Peor caso, mejor caso, media y dispersión)
# IMPORTANTE: Filtramos únicamente las columnas numéricas para calcular 'mean' y 'std'
# de forma segura sin generar TypeError por columnas de texto (como 'corner').
df_numerico = df_resultados.select_dtypes(include=[np.number])
df_resumen = df_numerico.agg(['min', 'max', 'mean', 'std']).T

print("==============================================================================")
print("--- RESUMEN PVT: MEJOR Y PEOR CASO DE PARÁMETROS DEL SELF-BIAS ---")
print("==============================================================================")
try:
    from IPython.display import display
    display(df_resumen)
except (NameError, ImportError):
    print(df_resumen)

# ==============================================================================
# 3. CÁLCULO DE FIGURAS DE MÉRITO DE INGENIERÍA
# ==============================================================================
print("\n==============================================================================")
print("--- FIGURAS DE MÉRITO CLAVE DEL DISEÑO SELF-BIAS ---")
print("==============================================================================")

col_vr = 'vr' if 'vr' in df_resultados.columns else ('vr_tran' if 'vr_tran' in df_resultados.columns else None)
col_ibias = 'ibias' if 'ibias' in df_resultados.columns else ('mp4_id_ua' if 'mp4_id_ua' in df_resultados.columns else None)

if col_vr:
    vr_nom = df_resultados[col_vr].mean()
    vr_min = df_resultados[col_vr].min()
    vr_max = df_resultados[col_vr].max()
    delta_vr = vr_max - vr_min
    temp_min = df_resultados['temperatura'].min()
    temp_max = df_resultados['temperatura'].max()
    delta_temp = temp_max - temp_min

    # Coeficiente de Temperatura en ppm/°C
    tc_vr = (delta_vr / (vr_nom * delta_temp)) * 1e6 if (vr_nom > 0 and delta_temp > 0) else 0

    print(f"• Voltaje de Referencia VR (Nominal / Media): {vr_nom*1000:.2f} mV")
    print(f"• Variación Total de VR [Min - Max]:        {vr_min*1000:.2f} mV a {vr_max*1000:.2f} mV (Δ = {delta_vr*1000:.2f} mV)")
    print(f"• Coeficiente Térmico de VR (TC):            {tc_vr:.1f} ppm/°C")

if col_ibias:
    ibias_nom = df_resultados[col_ibias].mean()
    ibias_min = df_resultados[col_ibias].min()
    ibias_max = df_resultados[col_ibias].max()
    delta_ibias = ibias_max - ibias_min

    print(f"• Corriente de Polarización Ibias (Media):   {ibias_nom:.2f} µA")
    print(f"• Variación Total de Ibias [Min - Max]:      {ibias_min:.2f} µA a {ibias_max:.2f} µA (Δ = {delta_ibias:.2f} µA)")

# ==============================================================================
# 4. VERIFICACIÓN DE SATURACIÓN DE TRANSISTORES (Vds - Vdsat > 0)
# ==============================================================================
cols_margin = [c for c in df_resultados.columns if c.endswith('_vds_margin')]
if cols_margin:
    print("\n--- Verificación de Márgenes de Saturación (Peor Caso Vds - Vdsat) ---")
    margenes_min = df_resultados[cols_margin].min()
    for col, val in margenes_min.items():
        trans_nombre = col.replace('_vds_margin', '').upper()
        estado = "✅ SATURACIÓN OK" if val > 0.05 else ("⚠️ SATURACIÓN MARGINAL" if val > 0 else "❌ REGIÓN TRIODO")
        print(f"• Transistor {trans_nombre:4s}: Margen Mínimo = {val*1000:7.1f} mV  -> {estado}")

# ==============================================================================
# 5. EXPORTACIÓN A EXCEL O CSV
# ==============================================================================
nombre_excel = "Reporte_Completo_SelfBias.xlsx"
try:
    with pd.ExcelWriter(nombre_excel) as writer:
        df_resultados.to_excel(writer, sheet_name="Corners_PVT")
        df_resumen.to_excel(writer, sheet_name="Resumen_Estadistico")
        if cols_margin:
            df_resultados[['temperatura', 'vdd', 'corner'] + cols_margin].to_excel(writer, sheet_name="Margenes_Saturacion")
    print(f"\n✅ ¡Reporte consolidado exportado exitosamente a '{nombre_excel}'!")

except (ImportError, ModuleNotFoundError):
    print("\nℹ️ Librería de Excel ('openpyxl') no detectada. Guardando resultados en formato CSV...")
    df_resultados.to_csv("Reporte_Corners_SelfBias.csv")
    df_resumen.to_csv("Resumen_Estadistico_SelfBias.csv")
    print("✅ ¡Tablas exportadas exitosamente a 'Reporte_Corners_SelfBias.csv' y 'Resumen_Estadistico_SelfBias.csv'!")



--- RESUMEN PVT: MEJOR Y PEOR CASO DE PARÁMETROS DEL SELF-BIAS ---


,min,max,mean,std
temperatura,-40.00000,125.000000,42.500000,84.643214
vdd,1.62000,1.980000,1.800000,0.184676
vr,0.59088,0.790689,0.688716,0.060494
ibias,6.77056,16.141500,11.033392,3.969747
vr_tran,0.59088,0.790686,0.688725,0.060496
temp,-40.00000,125.000000,42.500000,84.643214
vr_tan,0.59088,0.790686,0.688725,0.060496



--- FIGURAS DE MÉRITO CLAVE DEL DISEÑO SELF-BIAS ---
• Voltaje de Referencia VR (Nominal / Media): 688.72 mV
• Variación Total de VR [Min - Max]:        590.88 mV a 790.69 mV (Δ = 199.81 mV)
• Coeficiente Térmico de VR (TC):            1758.3 ppm/°C
• Corriente de Polarización Ibias (Media):   11.03 µA
• Variación Total de Ibias [Min - Max]:      6.77 µA a 16.14 µA (Δ = 9.37 µA)

ℹ️ Librería de Excel ('openpyxl') no detectada. Guardando resultados en formato CSV...
✅ ¡Tablas exportadas exitosamente a 'Reporte_Corners_SelfBias.csv' y 'Resumen_Estadistico_SelfBias.csv'!


In [5]:
# ==============================================================================
# ARCHIVO: self_bias_pvt.ipynb - Celda 5: Generador de Gráficas de Análisis PVT
# DESCRIPCIÓN:
# Genera curvas visuales de alta calidad para todas las figuras de mérito del Self-Bias:
# 1. Respuesta Transitoria de Arranque VR(t) superpuesta para todos los corners.
# 2. Señales internas de arranque VDD(t) y Vgp(t).
# 3. Sensibilidad Térmica: VR vs Temperatura por VDD y corner.
# 4. Sensibilidad Térmica: Ibias vs Temperatura.
# 5. Regulación de Línea: VR vs VDD.
# 6. Diagrama de Dispersión / Cajas por Esquina de Proceso.
# 7. Margen de Saturación de los transistores (Vds - Vdsat).
# Guarda cada figura en formato PNG a 300 DPI.
# ==============================================================================

print("\n📊 Iniciando generación de gráficas de caracterización PVT...")

# Configuración estética global de gráficos
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

# ------------------------------------------------------------------------------
# 1. RESPUESTA TRANSITORIA DE ARRANQUE: VR(t)
# ------------------------------------------------------------------------------
plt.figure(figsize=(10, 5))
exitosas_tran = 0
for llave, curvas in graficas_self_bias.items():
    if "time" in curvas and "vr" in curvas:
        x = np.real(curvas["time"]) * 1e6  # Convertir a microsegundos [µs]
        y = np.real(curvas["vr"])
        min_len = min(len(x), len(y))
        plt.plot(x[:min_len], y[:min_len], linewidth=0.8, alpha=0.35, color='#1f77b4')
        exitosas_tran += 1

plt.title(f"Análisis PVT: Respuesta Transitoria de Arranque VR(t) (Curvas: {exitosas_tran})", fontsize=12, fontweight='bold')
plt.xlabel("Tiempo [µs]", fontsize=10)
plt.ylabel("Voltaje de Referencia VR [V]", fontsize=10)
plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.savefig("self_bias_tran_vr.png", dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Guardada: 'self_bias_tran_vr.png' (Respuesta transitoria de VR)")

# ------------------------------------------------------------------------------
# 2. RESPUESTA TRANSITORIA DE SEÑALES INTERNAS: VDD(t) Y Vgp(t)
# ------------------------------------------------------------------------------
plt.figure(figsize=(10, 5))
exitosas_vgp = 0
for llave, curvas in graficas_self_bias.items():
    if "time" in curvas and "vgp" in curvas:
        x = np.real(curvas["time"]) * 1e6
        y_vgp = np.real(curvas["vgp"])
        min_len = min(len(x), len(y_vgp))
        plt.plot(x[:min_len], y_vgp[:min_len], linewidth=0.8, alpha=0.35, color='#2ca02c')
        exitosas_vgp += 1

plt.title(f"Análisis PVT: Dinámica de Tensión de Compuerta Vgp(t) (Curvas: {exitosas_vgp})", fontsize=12, fontweight='bold')
plt.xlabel("Tiempo [µs]", fontsize=10)
plt.ylabel("Tensión de Compuerta Vgp [V]", fontsize=10)
plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.savefig("self_bias_tran_vgp.png", dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Guardada: 'self_bias_tran_vgp.png' (Dinámica interna de polarización)")

# ------------------------------------------------------------------------------
# 3. SENSIBILIDAD TÉRMICA: VR vs TEMPERATURA
# ------------------------------------------------------------------------------
if 'temperatura' in df_resultados.columns and ('vr' in df_resultados.columns or 'vr_tran' in df_resultados.columns):
    col_v = 'vr' if 'vr' in df_resultados.columns else 'vr_tran'
    plt.figure(figsize=(9, 5))
    
    # Agrupar por VDD y corner para trazar líneas de sensibilidad
    for (vdd_val, corner_val), sub_df in df_resultados.groupby(['vdd', 'corner']):
        sub_ordenado = sub_df.sort_values('temperatura')
        plt.plot(sub_ordenado['temperatura'], sub_ordenado[col_v]*1000, marker='o', markersize=3, linewidth=1, alpha=0.6, label=f"VDD={vdd_val}V ({corner_val})")

    plt.title("Sensibilidad Térmica: Voltaje de Referencia VR vs Temperatura", fontsize=12, fontweight='bold')
    plt.xlabel("Temperatura [°C]", fontsize=10)
    plt.ylabel("Voltaje VR [mV]", fontsize=10)
    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    # Colocar leyenda si no son demasiadas combinaciones
    if len(df_resultados.groupby(['vdd', 'corner'])) <= 10:
        plt.legend(fontsize=8, loc='best')
    plt.tight_layout()
    plt.savefig("self_bias_vr_vs_temp.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("  ✓ Guardada: 'self_bias_vr_vs_temp.png' (Sensibilidad térmica de VR)")

# ------------------------------------------------------------------------------
# 4. SENSIBILIDAD TÉRMICA: Ibias vs TEMPERATURA
# ------------------------------------------------------------------------------
if 'temperatura' in df_resultados.columns and ('ibias' in df_resultados.columns or 'mp4_id_ua' in df_resultados.columns):
    col_i = 'ibias' if 'ibias' in df_resultados.columns else 'mp4_id_ua'
    plt.figure(figsize=(9, 5))
    
    for (vdd_val, corner_val), sub_df in df_resultados.groupby(['vdd', 'corner']):
        sub_ordenado = sub_df.sort_values('temperatura')
        plt.plot(sub_ordenado['temperatura'], sub_ordenado[col_i], marker='s', markersize=3, linewidth=1, alpha=0.6, label=f"VDD={vdd_val}V ({corner_val})")

    plt.title("Sensibilidad Térmica: Corriente de Polarización Ibias vs Temperatura", fontsize=12, fontweight='bold')
    plt.xlabel("Temperatura [°C]", fontsize=10)
    plt.ylabel("Corriente Ibias [µA]", fontsize=10)
    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    if len(df_resultados.groupby(['vdd', 'corner'])) <= 10:
        plt.legend(fontsize=8, loc='best')
    plt.tight_layout()
    plt.savefig("self_bias_ibias_vs_temp.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("  ✓ Guardada: 'self_bias_ibias_vs_temp.png' (Sensibilidad térmica de Ibias)")

# ------------------------------------------------------------------------------
# 5. REGULACIÓN DE LÍNEA: VR vs VDD
# ------------------------------------------------------------------------------
if 'vdd' in df_resultados.columns and ('vr' in df_resultados.columns or 'vr_tran' in df_resultados.columns):
    col_v = 'vr' if 'vr' in df_resultados.columns else 'vr_tran'
    plt.figure(figsize=(9, 5))
    
    for (temp_val, corner_val), sub_df in df_resultados.groupby(['temperatura', 'corner']):
        sub_ordenado = sub_df.sort_values('vdd')
        plt.plot(sub_ordenado['vdd'], sub_ordenado[col_v]*1000, marker='^', markersize=4, linewidth=1, alpha=0.6, label=f"T={temp_val}°C ({corner_val})")

    plt.title("Regulación de Línea: Voltaje de Referencia VR vs Alimentación VDD", fontsize=12, fontweight='bold')
    plt.xlabel("Alimentación VDD [V]", fontsize=10)
    plt.ylabel("Voltaje VR [mV]", fontsize=10)
    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    if len(df_resultados.groupby(['temperatura', 'corner'])) <= 10:
        plt.legend(fontsize=8, loc='best')
    plt.tight_layout()
    plt.savefig("self_bias_vr_vs_vdd.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("  ✓ Guardada: 'self_bias_vr_vs_vdd.png' (Regulación de línea)")

# ------------------------------------------------------------------------------
# 6. DISPERSIÓN POR CORNER DE PROCESO (BOXPLOT)
# ------------------------------------------------------------------------------
if 'corner' in df_resultados.columns and ('vr' in df_resultados.columns or 'vr_tran' in df_resultados.columns):
    col_v = 'vr' if 'vr' in df_resultados.columns else 'vr_tran'
    plt.figure(figsize=(9, 5))
    
    corners_unicos = df_resultados['corner'].unique()
    datos_por_corner = [df_resultados[df_resultados['corner'] == c][col_v]*1000 for c in corners_unicos]

    plt.boxplot(datos_por_corner, labels=corners_unicos, patch_artist=True,
                boxprops=dict(facecolor='#d9edf7', color='#31708f'),
                medianprops=dict(color='#d9534f', linewidth=1.5))
    
    plt.title("Dispersión de Voltaje VR agrupado por Esquina de Proceso Sky130", fontsize=12, fontweight='bold')
    plt.xlabel("Corner de Proceso", fontsize=10)
    plt.ylabel("Voltaje VR [mV]", fontsize=10)
    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    plt.tight_layout()
    plt.savefig("self_bias_dispersion_corners.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("  ✓ Guardada: 'self_bias_dispersion_corners.png' (Dispersión por corner)")

# ------------------------------------------------------------------------------
# 7. MÁRGENES DE SATURACIÓN DE TRANSISTORES (Vds - Vdsat)
# ------------------------------------------------------------------------------
cols_margin = [c for c in df_resultados.columns if c.endswith('_vds_margin')]
if cols_margin:
    plt.figure(figsize=(10, 5))
    nombres_trans = [c.replace('_vds_margin', '').upper() for c in cols_margin]
    peores_margenes = df_resultados[cols_margin].min() * 1000  # Convertir a mV
    
    colores = ['#2ca02c' if m > 50 else ('#ff7f0e' if m > 0 else '#d62728') for m in peores_margenes]
    barras = plt.bar(nombres_trans, peores_margenes, color=colores, edgecolor='#333333', linewidth=0.8)

    plt.axhline(0, color='red', linestyle='--', linewidth=1, label="Límite Saturación / Triodo (0 mV)")
    plt.axhline(50, color='gray', linestyle=':', linewidth=0.8, label="Margen Seguro Recomendado (50 mV)")
    
    # Etiquetas de valor sobre cada barra
    for barra in barras:
        altura = barra.get_height()
        pos_y = altura + 3 if altura >= 0 else altura - 10
        plt.text(barra.get_x() + barra.get_width()/2., pos_y, f"{altura:.0f} mV",
                 ha='center', va='bottom' if altura >= 0 else 'top', fontsize=9)

    plt.title("Peor Margen de Saturación (Vds - Vdsat) para Transistores del Self-Bias", fontsize=12, fontweight='bold')
    plt.xlabel("Transistor", fontsize=10)
    plt.ylabel("Margen Mínimo Vds - Vdsat [mV]", fontsize=10)
    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7, axis='y')
    plt.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("self_bias_margen_saturacion.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("  ✓ Guardada: 'self_bias_margen_saturacion.png' (Márgenes de saturación)")

print("\n✨ ¡Todas las gráficas de caracterización PVT fueron generadas y guardadas exitosamente!")




📊 Iniciando generación de gráficas de caracterización PVT...
  ✓ Guardada: 'self_bias_tran_vr.png' (Respuesta transitoria de VR)
  ✓ Guardada: 'self_bias_tran_vgp.png' (Dinámica interna de polarización)
  ✓ Guardada: 'self_bias_vr_vs_temp.png' (Sensibilidad térmica de VR)
  ✓ Guardada: 'self_bias_ibias_vs_temp.png' (Sensibilidad térmica de Ibias)
  ✓ Guardada: 'self_bias_vr_vs_vdd.png' (Regulación de línea)
  ✓ Guardada: 'self_bias_dispersion_corners.png' (Dispersión por corner)

✨ ¡Todas las gráficas de caracterización PVT fueron generadas y guardadas exitosamente!


/tmp/ipykernel_57090/518837920.py:144: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(datos_por_corner, labels=corners_unicos, patch_artist=True,
